# Lecture 1: Learning and Mathematical Preliminaries

This notebook contains worked examples and computational demonstrations for the companion slides.

[Open in Google Colab](https://colab.research.google.com/github/DrFarrukh/teaching/blob/main/courses/artificial-neural-networks/lectures/lecture-01/notebook.ipynb). Upload the notebook to Google Colab and select **Runtime → Run all**. The examples use synthetic data and need no file uploads.


In [ ]:
# Google Colab includes these packages in its standard runtime.
# Uncomment this line only if a package is missing:
# %pip install torch numpy matplotlib


Tensor

In [13]:
import torch

X = torch.tensor([
    [0.12, -0.03],
    [0.16, -0.01],
    [0.09,  0.02]
], dtype=torch.float32)

print(X)

tensor([[ 0.1200, -0.0300],
        [ 0.1600, -0.0100],
        [ 0.0900,  0.0200]])


In [16]:
print(X.shape)
print(X.dtype)
print(X.numel())

torch.Size([3, 2])
torch.float32
6


In [17]:
print(X[0])       # first row, shape [2]

tensor([ 0.1200, -0.0300])


In [18]:
print(X[:, 1])    # second column, shape [3]

tensor([-0.0300, -0.0100,  0.0200])


In [19]:
print(X[1, 0])    # one scalar

tensor(0.1600)


In [20]:
print(X**2)       # elementwise square

tensor([[1.4400e-02, 9.0000e-04],
        [2.5600e-02, 1.0000e-04],
        [8.1000e-03, 4.0000e-04]])


In [22]:
M = torch.tensor([
    [0.5,   -0.11],
    [-0.05,  0.19],
    [-0.01,  0.12]
], dtype=torch.float32)
print(M)

tensor([[ 0.5000, -0.1100],
        [-0.0500,  0.1900],
        [-0.0100,  0.1200]])


In [23]:
X*M

tensor([[ 0.0600,  0.0033],
        [-0.0080, -0.0019],
        [-0.0009,  0.0024]])

In [24]:
X@M

RuntimeError: mat1 and mat2 shapes cannot be multiplied (3x2 and 3x2)

In [25]:
X@M.T

tensor([[ 0.0633, -0.0117, -0.0048],
        [ 0.0811, -0.0099, -0.0028],
        [ 0.0428, -0.0007,  0.0015]])

In [26]:
del X

In [27]:
X

NameError: name 'X' is not defined

Reductions

In [28]:
X = torch.tensor([
    [1, 2, 3],
    [4, 5, 6],
    ], dtype=torch.float32)
print(X)

tensor([[1., 2., 3.],
        [4., 5., 6.]])


In [29]:
X.sum()                 # scalar, shape []

tensor(21.)

In [30]:
X.sum(dim=0)            # shape [3]

tensor([5., 7., 9.])

In [31]:
X.sum(dim=1)            # shape [2]

tensor([ 6., 15.])

In [32]:
X.mean(dim=1, keepdim=True)  # shape [2, 1]

tensor([[2.],
        [5.]])

Broadcasting

In [35]:
X = torch.tensor([[1., 10.],
                  [20., 10.],
                  [3., 30.]])

mu = X.mean(dim=0)
print(mu)

tensor([ 8.0000, 16.6667])


In [36]:
X_centered = X - mu

print(X_centered)
print(X_centered.mean(dim=0))


tensor([[-7.0000, -6.6667],
        [12.0000, -6.6667],
        [-5.0000, 13.3333]])
tensor([0.0000e+00, 6.3578e-07])


In [37]:
print(X.shape)
print(mu.shape)

torch.Size([3, 2])
torch.Size([2])


Matrix-Matrix Multiplication

In [38]:
A = torch.tensor([[1., 2.],
                  [3., 4.]])
B = torch.tensor([[2.,  0.],
                  [1., -1.]])

print(A @ B)

tensor([[ 4., -2.],
        [10., -4.]])


Y_hat = X @ W.T + b

Norms

In [40]:
x = torch.tensor([3., -4.])
torch.linalg.vector_norm(x, ord=1)  # 7

tensor(7.)

In [41]:
torch.linalg.vector_norm(x, ord=2)  # 5

tensor(5.)

Autograd

In [42]:
x = torch.tensor(2.0)
y = torch.tensor(5.0)
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [43]:
print(x)
print(y)
print(w)
print(b)

tensor(2.)
tensor(5.)
tensor(1., requires_grad=True)
tensor(0., requires_grad=True)


In [44]:
y_hat = w*x + b
loss = (y_hat - y)**2
loss.backward()

In [45]:
print(loss.item())
print(w.grad)
print(b.grad)

9.0
tensor(-12.)
tensor(-6.)


* **requires_grad=True** tells PyTorch to record
operations involving that tensor.
* the forward pass constructs a computational graph while computing values;
* **backward()** traverses the graph in reverse and applies the chain rule;
* gradients accumulate in **.grad**.

In [46]:
w.grad.zero_()
b.grad.zero_()

tensor(0.)

Finite-difference check

In [ ]:
def loss_fn(w):
    return (2*w - 5)**2

w0 = 1.0
h = 1e-4
g_fd = (loss_fn(w0 + h) - loss_fn(w0)) / h
print(g_fd)  # approximately -12


In [ ]:
g_cd = (loss_fn(w0 + h) - loss_fn(w0 - h)) / (2*h)
g_cd

In [ ]:
torch.manual_seed(7)
rolls = torch.randint(1, 7, (200,))

running_frequency = (rolls == 1).float().cumsum(0)
running_frequency /= torch.arange(1, 201)

print(running_frequency[-1])
print(1/6)


Complete PyTorch implementation

In [47]:
import torch

x = torch.tensor([1., 2., 3.])
y = 2*x + 1

w = torch.tensor(0., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
eta = 0.1

# Forward pass
y_hat = w*x + b
loss = ((y_hat - y)**2).mean()

# Reverse-mode automatic differentiation
loss.backward()

print("before:", w.item(), b.item(), loss.item())
print("grads :", w.grad.item(), b.grad.item())

# Parameter update must not itself be recorded in the graph
with torch.no_grad():
    w -= eta*w.grad
    b -= eta*b.grad

# Clear accumulated gradients before another backward pass
w.grad.zero_()
b.grad.zero_()

# Evaluate the updated parameters
y_hat_new = w*x + b
loss_new = ((y_hat_new - y)**2).mean()

print("after :", w.item(), b.item(), loss_new.item())
print("preds :", y_hat_new)


before: 0.0 0.0 27.66666603088379
grads : -22.666667938232422 -10.0
after : 2.266666889190674 1.0 0.33185240626335144
preds : tensor([3.2667, 5.5333, 7.8000], grad_fn=<AddBackward0>)


Preprocessing & Leakage

In [ ]:
X_train = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0]
], dtype=torch.float32)

X_val = torch.tensor([
    [10.0, 11.0, 12.0]
], dtype=torch.float32)

X_test = torch.tensor([
    [13.0, 14.0, 15.0],
    [16.0, 17.0, 18.0]
], dtype=torch.float32)

print("Original X_train:\n", X_train)
print("Original X_val:\n", X_val)
print("Original X_test:\n", X_test)

In [ ]:
mu = X_train.mean(dim=0)
sigma = X_train.std(dim=0).clamp_min(1e-8)

X_train_scaled = (X_train - mu) / sigma
X_val_scaled   = (X_val   - mu) / sigma
X_test_scaled  = (X_test  - mu) / sigma

print("\nScaled X_train:\n", X_train_scaled)
print("\nScaled X_val:\n", X_val_scaled)
print("\nScaled X_test:\n", X_test_scaled)

In [ ]:
def loss_fn(w):
    return (2 * w - 5) ** 2


w0 = 1.0
h = 1e-4

# Forward-difference
g_fd = (loss_fn(w0 + h) - loss_fn(w0)) / h
print(f"Forward Difference: {g_fd}")  # Output: -11.999599999991823

# Central-difference
g_cd = (loss_fn(w0 + h) - loss_fn(w0 - h)) / (2 * h)
print(f"Central Difference: {g_cd}")  # Output: -12.000000000005339


Illustration of Complete Cycle

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

torch.manual_seed(7)

plt.rcParams.update({
    "figure.figsize": (9, 5.5),
    "figure.dpi": 110,
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA_COLOR = "tab:blue"
INITIAL_COLOR = "tab:red"
UPDATED_COLOR = "tab:green"
RESIDUAL_COLOR = "tab:orange"


def finish_plot(ax, title, x_label="input x", y_label="target / prediction"):
    """Apply consistent labels and display the current figure."""
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()



In [ ]:
# %% 1. Create a tiny regression dataset
# The unknown relationship used to generate the observations is y = 2x + 1.
x = torch.tensor([1.0, 2.0, 3.0])
y = 2 * x + 1

print("x =", x)
print("y =", y)

fig, ax = plt.subplots()
ax.scatter(x.numpy(), y.numpy(), s=90, color=DATA_COLOR, label="observed data", zorder=3)

for xi, yi in zip(x, y):
    ax.annotate(
        f"({xi.item():.0f}, {yi.item():.0f})",
        (xi.item(), yi.item()),
        xytext=(7, 7),
        textcoords="offset points",
    )

ax.set_xlim(0.5, 3.5)
ax.set_ylim(-0.5, 8.5)
finish_plot(ax, "Step 1 — The observed training data")


In [ ]:
# %% 2. Initialize the model and perform the forward pass
# Model: y_hat = w*x + b
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.1

y_hat_initial = w * x + b
loss_initial = ((y_hat_initial - y) ** 2).mean()

print(f"initial w    = {w.item():.3f}")
print(f"initial b    = {b.item():.3f}")
print("initial yhat =", y_hat_initial.detach())
print(f"initial loss = {loss_initial.item():.3f}")

fig, ax = plt.subplots()
ax.scatter(x.numpy(), y.numpy(), s=90, color=DATA_COLOR, label="observed data", zorder=3)

x_line = np.linspace(0.5, 3.5, 200)
y_line_initial = w.item() * x_line + b.item()
ax.plot(
    x_line,
    y_line_initial,
    linewidth=3,
    color=INITIAL_COLOR,
    label=r"initial model: $\hat{y}=0$",
)

# Residuals are the vertical distances between prediction and target.
for xi, yi, yhi in zip(x, y, y_hat_initial.detach()):
    ax.plot(
        [xi.item(), xi.item()],
        [yhi.item(), yi.item()],
        linestyle="--",
        linewidth=2,
        color=RESIDUAL_COLOR,
    )

ax.plot([], [], "--", linewidth=2, color=RESIDUAL_COLOR, label="residuals")
ax.text(
    0.98,
    0.06,
    f"MSE = {loss_initial.item():.3f}",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=13,
)
ax.set_xlim(0.5, 3.5)
ax.set_ylim(-0.5, 8.5)
finish_plot(ax, "Step 2 — Initial predictions and their errors")

In [ ]:
# %% 3. Visualize the loss as a function of both parameters
# For every possible pair (w, b), calculate the mean squared loss.
w_values = np.linspace(-1.0, 4.5, 220)
b_values = np.linspace(-2.0, 4.0, 220)
W, B = np.meshgrid(w_values, b_values)

x_np = x.numpy()
y_np = y.numpy()

predictions_over_grid = W[..., None] * x_np + B[..., None]
loss_surface = np.mean((predictions_over_grid - y_np) ** 2, axis=-1)

fig, ax = plt.subplots(figsize=(9, 6))
filled = ax.contourf(W, B, loss_surface, levels=30, cmap="viridis")
contours = ax.contour(
    W,
    B,
    loss_surface,
    levels=[0.25, 0.5, 1, 2, 5, 10, 20, 40],
    colors="white",
    linewidths=0.8,
    alpha=0.75,
)
ax.clabel(contours, inline=True, fontsize=9, fmt="%g")
fig.colorbar(filled, ax=ax, label="mean squared loss")

ax.scatter(0, 0, s=120, color=INITIAL_COLOR, edgecolor="white", label="start: (0, 0)", zorder=5)
ax.scatter(2, 1, s=160, marker="*", color="gold", edgecolor="black", label="minimum: (2, 1)", zorder=5)

finish_plot(
    ax,
    "Step 3 — The loss landscape",
    x_label="weight w",
    y_label="bias b",
)


In [ ]:


# %% 4. Backpropagate, inspect the gradient, and update the parameters
loss_initial.backward()

dw = w.grad.item()
db = b.grad.item()

print(f"dL/dw = {dw:.3f}")
print(f"dL/db = {db:.3f}")
print(f"negative gradient direction = ({-dw:.3f}, {-db:.3f})")

w_before = w.item()
b_before = b.item()

with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad

w_after = w.item()
b_after = b.item()

print(f"updated w = {w_after:.3f}")
print(f"updated b = {b_after:.3f}")

fig, ax = plt.subplots(figsize=(9, 6))
filled = ax.contourf(W, B, loss_surface, levels=30, cmap="viridis")
contours = ax.contour(
    W,
    B,
    loss_surface,
    levels=[0.25, 0.5, 1, 2, 5, 10, 20, 40],
    colors="white",
    linewidths=0.8,
    alpha=0.75,
)
ax.clabel(contours, inline=True, fontsize=9, fmt="%g")
fig.colorbar(filled, ax=ax, label="mean squared loss")

ax.scatter(
    w_before,
    b_before,
    s=120,
    color=INITIAL_COLOR,
    edgecolor="white",
    label=f"before: ({w_before:.3f}, {b_before:.3f})",
    zorder=5,
)
ax.scatter(
    w_after,
    b_after,
    s=120,
    color=UPDATED_COLOR,
    edgecolor="white",
    label=f"after: ({w_after:.3f}, {b_after:.3f})",
    zorder=5,
)
ax.scatter(2, 1, s=160, marker="*", color="gold", edgecolor="black", label="minimum: (2, 1)", zorder=5)

# The arrow is exactly the gradient-descent parameter update.
ax.annotate(
    "",
    xy=(w_after, b_after),
    xytext=(w_before, b_before),
    arrowprops={"arrowstyle": "->", "linewidth": 3, "color": "tab:red"},
)
ax.text(
    (w_before + w_after) / 2,
    (b_before + b_after) / 2 + 0.25,
    r"$-\eta\nabla L$",
    color="white",
    ha="center",
    fontsize=13,
    weight="bold",
)

finish_plot(
    ax,
    "Step 4 — One gradient-descent update on the loss landscape",
    x_label="weight w",
    y_label="bias b",
)


In [ ]:


# %% 5. Recompute predictions using the updated parameters
# Clear gradients before another backward pass.
w.grad.zero_()
b.grad.zero_()

y_hat_updated = w * x + b
loss_updated = ((y_hat_updated - y) ** 2).mean()

print("updated yhat =", y_hat_updated.detach())
print(f"updated loss = {loss_updated.item():.3f}")

fig, ax = plt.subplots()
ax.scatter(x.numpy(), y.numpy(), s=90, color=DATA_COLOR, label="observed data", zorder=4)

ax.plot(
    x_line,
    w_before * x_line + b_before,
    linestyle="--",
    linewidth=2,
    color=INITIAL_COLOR,
    alpha=0.75,
    label="initial model",
)
ax.plot(
    x_line,
    w_after * x_line + b_after,
    linewidth=3,
    color=UPDATED_COLOR,
    label=rf"updated model: $\hat{{y}}={w_after:.3f}x+{b_after:.3f}$",
)

for xi, yi, yhi in zip(x, y, y_hat_updated.detach()):
    ax.plot(
        [xi.item(), xi.item()],
        [yhi.item(), yi.item()],
        linestyle="--",
        linewidth=2,
        color=RESIDUAL_COLOR,
    )

ax.plot([], [], "--", linewidth=2, color=RESIDUAL_COLOR, label="new residuals")
ax.text(
    0.98,
    0.06,
    f"MSE = {loss_updated.item():.3f}",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=13,
)
ax.set_xlim(0.5, 3.5)
ax.set_ylim(-0.5, 8.5)
finish_plot(ax, "Step 5 — Predictions after one update")


In [ ]:


# %% 6. Compare the loss before and after the update
loss_values = [loss_initial.item(), loss_updated.item()]
labels = ["before update", "after one update"]
colors = [INITIAL_COLOR, UPDATED_COLOR]

fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.bar(labels, loss_values, color=colors, width=0.55)

for bar, value in zip(bars, loss_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.6,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=14,
        weight="bold",
    )

ax.set_ylim(0, max(loss_values) * 1.18)
ax.set_ylabel("mean squared loss")
ax.set_title("Step 6 — One gradient step greatly reduces the training loss")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

print("\nComplete learning pass")
print("----------------------")
print(f"parameters: ({w_before:.3f}, {b_before:.3f}) -> ({w_after:.3f}, {b_after:.3f})")
print(f"loss:       {loss_initial.item():.3f} -> {loss_updated.item():.3f}")
print("Important: lower training loss does not by itself prove generalization.")



In [ ]:

# %% 7. Optional: verify the autograd gradient using finite differences
def numpy_loss(weight, bias):
    predictions = weight * x_np + bias
    return np.mean((predictions - y_np) ** 2)


h = 1e-4
dw_finite = (numpy_loss(w_before + h, b_before) - numpy_loss(w_before, b_before)) / h
db_finite = (numpy_loss(w_before, b_before + h) - numpy_loss(w_before, b_before)) / h

print(f"autograd dL/dw        = {dw:.6f}")
print(f"finite-difference dL/dw = {dw_finite:.6f}")
print(f"autograd dL/db        = {db:.6f}")
print(f"finite-difference dL/db = {db_finite:.6f}")


In [ ]:

# %% 8. Optional: interactive, rotatable 3D loss surface
# Plotly provides MATLAB-like mouse rotation, scroll zoom, pan, and hover values.
# Install once if necessary: pip install plotly
import plotly.graph_objects as go


loss_at_start = numpy_loss(w_before, b_before)
loss_after_step = numpy_loss(w_after, b_after)
loss_at_minimum = numpy_loss(2.0, 1.0)

# Downsampling keeps interaction smooth while preserving the surface shape.
skip = 3

surface = go.Surface(
    x=W[::skip, ::skip],
    y=B[::skip, ::skip],
    z=loss_surface[::skip, ::skip],
    colorscale="Viridis",
    opacity=0.82,
    colorbar={"title": "MSE"},
    hovertemplate=(
        "w=%{x:.3f}<br>"
        "b=%{y:.3f}<br>"
        "loss=%{z:.3f}<extra></extra>"
    ),
    name="loss surface",
    showscale=True,
)

# One trace draws the complete parameter movement from start to updated value.
update_path = go.Scatter3d(
    x=[w_before, w_after],
    y=[b_before, b_after],
    z=[loss_at_start, loss_after_step],
    mode="lines+markers+text",
    line={"color": "crimson", "width": 8},
    marker={"size": 7, "color": ["crimson", "limegreen"]},
    text=["start", "after one step"],
    textposition="top center",
    hovertemplate=(
        "%{text}<br>"
        "w=%{x:.3f}<br>"
        "b=%{y:.3f}<br>"
        "loss=%{z:.3f}<extra></extra>"
    ),
    name="gradient update",
)

minimum = go.Scatter3d(
    x=[2.0],
    y=[1.0],
    z=[loss_at_minimum],
    mode="markers+text",
    marker={"size": 9, "symbol": "diamond", "color": "gold"},
    text=["minimum"],
    textposition="bottom center",
    hovertemplate=(
        "minimum<br>w=%{x:.3f}<br>b=%{y:.3f}<br>loss=%{z:.3f}"
        "<extra></extra>"
    ),
    name="minimum",
)

fig_interactive = go.Figure(data=[surface, update_path, minimum])
fig_interactive.update_layout(
    title="Interactive loss surface over the (w, b) meshgrid",
    scene={
        "xaxis_title": "weight w",
        "yaxis_title": "bias b",
        "zaxis_title": "mean squared loss",
        "camera": {"eye": {"x": 1.15, "y": -1.55, "z": 1.15}},
        "aspectmode": "auto",
    },
    legend={"x": 0.01, "y": 0.99},
    margin={"l": 0, "r": 0, "b": 0, "t": 55},
)

# Drag to rotate. Scroll to zoom. Double-click to reset the camera.
fig_interactive.show(config={"scrollZoom": True, "displaylogo": False})

# %%
